[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/04_diseno_tokenizers_modernos/04_proyecto_diseno_tokenizers_modernos.ipynb)

# Proyecto 04 · Benchmark de fertilidad/paridad y diseño del vocabulario de Políglota

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | ≈ 2 h |
| **GPU** | Ninguna (CPU, `FAST_DEV_RUN = True`) |
| **Prerrequisitos** | Lección del módulo 04 y proyecto 03 |

## Contexto de negocio
Políglota Labs va a **entrenar su propio embedder** y necesita congelar el vocabulario. Tu CTO pide dos entregables: **(A)** un benchmark honesto de los tokenizers candidatos (los públicos y el tuyo) en **ES / EN / código** y en otros idiomas con texto paralelo,
y **(B)** una **receta de vocabulario** (mezcla de corpus + tamaño $V$) justificada con una función de coste con el tráfico esperado (50 % ES, 30 % EN, 20 % código).

> **Conexión.** Viene del proyecto 03 (entrenaste y comparaste algoritmos con tubería idéntica). Va hacia el módulo 05 (cómo falla un vocabulario) y, más adelante, hacia el embedder de Políglota (módulos 09–11), que **hereda** este vocabulario.

## Dataset
Corpus Políglota (`carga_corpus`: HF → espejo Europarl + stdlib → sintético) y texto paralelo (`carga_paralelo`: FLORES+ → opus-100 → 16 frases × 12 idiomas escritas a mano; avisa en pantalla).

## Entregables y rúbrica
1. `benchmark(registro, par)`: tabla de **prima de paridad** y **bytes/token** por tokenizer e idioma (+ `tokens_por_kb` en ES/EN/código).
2. `entrena_candidato(prop, V)` y `evalua_candidato(tok)` → tokens por KB por idioma.
3. `selecciona(...)`: barrido de mezclas × V y elección por la función de coste $J$.
4. Un archivo `poliglota_vocab.json` con la receta y las métricas, y una decisión escrita.

La rúbrica puntúa de forma automática. Umbrales calibrados con **lo que mide la solución de referencia** (corpus espejo, `FAST_DEV_RUN=True`, mini-conjunto paralelo) menos un margen; los criterios *relativos* se sostienen con otros datos, los absolutos se relajan.

In [ ]:
!pip install -q -U tokenizers sentencepiece tiktoken datasets matplotlib seaborn scikit-learn

In [ ]:
import os, sys, re, math, json, random, time, unicodedata, collections, itertools, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")          # evita sobre-suscripción de hilos
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU, minutos). False: escala completa (Colab Pro)
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
COL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "rojo": "#D62728"}
try:
    import torch
    torch.set_num_threads(1); torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN)

In [ ]:
# ---- Utilidades de corpus Políglota (copia local; contrato: carga_corpus(fast_dev) -> dict) ----
import urllib.request, urllib.error, zipfile, io, inspect, ast

def hay_red(url: str, timeout: float = 6.0) -> bool:
    """Sonda rápida para no esperar minutos si el host está bloqueado (proxy corporativo, etc.)."""
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=timeout)
        return True
    except urllib.error.HTTPError:
        return True          # el servidor respondió (aunque sea 4xx): hay ruta de red
    except Exception:
        return False         # túnel bloqueado, DNS, timeout...

def _hf_wikipedia(lang: str, n: int, max_chars: int):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    out = []
    for ex in ds:
        if len(ex["text"]) > 400:
            out.append({"id": f"wiki_{lang}_{len(out)}", "lang": lang, "text": ex["text"][:max_chars]})
        if len(out) >= n:
            break
    return out

def _hf_codesearchnet(n: int):
    from datasets import load_dataset
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("claudios/code_search_net", {"name": "python"}),
                     ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw)
            out = []
            for ex in ds:
                code = ex.get("func_code_string") or ex.get("code") or ex.get("original_string")
                if code and 80 < len(code) < 3000:
                    out.append({"id": f"csn_{len(out)}", "lang": "code", "text": code})
                if len(out) >= n:
                    return out
        except Exception as e:
            print("   (CodeSearchNet vía", repo, "falló:", type(e).__name__, ")")
    raise RuntimeError("CodeSearchNet no disponible")

def _detok(s: str) -> str:
    s = re.sub(r"\s+([,.;:!?%)\]])", r"\1", s)
    s = re.sub(r"([(\[¿¡])\s+", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
def _espejo_nltk(n: int):
    """Espejo público: muestra de Europarl (ES/EN) del repositorio nltk_data en GitHub (CC/uso de investigación)."""
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/europarl_raw.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
    docs = []
    for lang, folder, ext in [("es", "spanish", ".es"), ("en", "english", ".en")]:
        lines = []
        for name in sorted(z.namelist()):
            if f"/{folder}/" in name and name.endswith(ext):
                lines += [_detok(l) for l in z.read(name).decode("utf-8", "ignore").splitlines() if len(l.split()) > 6]
        step = 8
        chunks = [" ".join(lines[i:i + step]) for i in range(0, len(lines), step)]
        random.Random(SEED).shuffle(chunks)
        docs += [{"id": f"ep_{lang}_{i}", "lang": lang, "text": c[:2000]} for i, c in enumerate(chunks[:n])]
    return docs

def _codigo_stdlib(n: int):
    """Código Python real de la biblioteca estándar (siempre disponible, sin red)."""
    import json, re as _re, argparse, textwrap, collections as _c, itertools as _i, functools, pathlib, csv, heapq, bisect
    import statistics, difflib, string, shutil, tempfile, fractions, random as _r, datetime, calendar, glob, fnmatch, logging
    out = []
    for mod in [json.decoder, json.encoder, _re, argparse, textwrap, functools, pathlib, csv, heapq, bisect, statistics, difflib,
                shutil, tempfile, fractions, datetime, calendar, glob, fnmatch, logging, _c, _r]:
        try:
            tree = ast.parse(inspect.getsource(mod)); src = inspect.getsource(mod).splitlines()
        except (OSError, TypeError, SyntaxError):
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef,)) and 4 <= (node.end_lineno - node.lineno) <= 40:
                code = "\n".join(src[node.lineno - 1: node.end_lineno])
                if 80 < len(code) < 3000:
                    out.append({"id": f"std_{len(out)}", "lang": "code", "text": code})
    random.Random(SEED).shuffle(out)
    return out[:n]

In [ ]:
_ES_S = ["el sistema", "la base de datos", "el servidor", "la aplicación", "el modelo", "la biblioteca", "el usuario", "la función", "el módulo", "la consulta"]
_ES_V = ["procesa", "almacena", "devuelve", "actualiza", "valida", "descarga", "ejecuta", "registra", "indexa", "compara"]
_ES_O = ["los documentos relevantes", "una lista de resultados", "los parámetros de configuración", "las peticiones entrantes", "la información almacenada",
         "varias tablas de forma eficiente", "los errores más frecuentes", "las versiones anteriores", "los textos en diferentes idiomas", "un conjunto de datos nuevo"]
_ES_A = ["rápidamente", "automáticamente", "correctamente", "de manera incremental", "sin interrupciones", "cada vez que cambia la configuración", "antes de la instalación", "durante la inicialización"]
_EN_S = ["the system", "the database", "the server", "the application", "the model", "the library", "the user", "the function", "the module", "the query"]
_EN_V = ["processes", "stores", "returns", "updates", "validates", "downloads", "executes", "logs", "indexes", "compares"]
_EN_O = ["the relevant documents", "a list of results", "the configuration parameters", "the incoming requests", "the stored information",
         "several tables efficiently", "the most frequent errors", "the previous versions", "texts in different languages", "a new dataset"]
_EN_A = ["quickly", "automatically", "correctly", "incrementally", "without interruptions", "whenever the configuration changes", "before the installation", "during initialization"]

def _sintetico(n: int):
    """Último recurso: texto ES/EN generado por plantillas + funciones Python plantilla (mismo esquema)."""
    rng = random.Random(SEED); docs = []
    for lang, S, V, O, A in [("es", _ES_S, _ES_V, _ES_O, _ES_A), ("en", _EN_S, _EN_V, _EN_O, _EN_A)]:
        for i in range(n):
            sents = []
            for _ in range(rng.randint(4, 8)):
                s = f"{rng.choice(S)} {rng.choice(V)} {rng.choice(O)} {rng.choice(A)}."
                sents.append(s[0].upper() + s[1:])
            docs.append({"id": f"syn_{lang}_{i}", "lang": lang, "text": " ".join(sents)})
    names = ["load", "parse", "merge", "filter", "count", "split", "encode", "decode", "score", "rank"]
    things = ["items", "tokens", "rows", "words", "records", "paths", "values", "pairs"]
    for i in range(n):
        f, t = rng.choice(names), rng.choice(things)
        docs.append({"id": f"syn_code_{i}", "lang": "code", "text":
            f"def {f}_{t}({t}, limit={rng.randint(2, 99)}):\n    \"\"\"{f.capitalize()} the {t} and return at most limit results.\"\"\"\n"
            f"    result = []\n    for x in {t}:\n        if len(result) >= limit:\n            break\n        result.append(x)\n    return result\n"})
    return docs

In [ ]:
def _mini_golden(docs, n_q: int = 24):
    """Mini golden set: la consulta es la primera frase (≤ 12 palabras) del documento; qrels {qid: {doc_id: 1}}."""
    rng = random.Random(SEED); queries, qrels = [], {}
    for lang in ["es", "en", "code"]:
        pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool)
        for d in pool[: n_q // 3]:
            if lang == "code":
                try:
                    q = (ast.get_docstring(ast.parse(d["text"]).body[0]) or "").split("\n")[0]
                except SyntaxError:
                    q = ""
                q = q or re.sub(r"[_(].*", "", d["text"].split("def ")[-1]).replace("_", " ")
            else:
                q = " ".join(re.split(r"(?<=[.!?])\s", d["text"])[0].split()[:12])
            qid = f"q{len(queries)}"; queries.append({"qid": qid, "lang": lang, "text": q}); qrels[qid] = {d["id"]: 1}
    return queries, qrels

In [ ]:
def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota. Cadena: oficial (HF Wikipedia + CodeSearchNet) -> espejo (Europarl vía GitHub + stdlib) -> sintético.
    Devuelve {"docs", "queries", "qrels", "source": "real"|"synthetic", "detail": {...}}."""
    n = 150 if fast_dev else 3000
    max_chars = 2000 if fast_dev else 6000
    docs, detail = [], {}
    hf_ok = hay_red("https://huggingface.co")
    for lang in ["es", "en"]:
        got = None
        if hf_ok:
            try: got, detail[lang] = _hf_wikipedia(lang, n, max_chars), "hf:wikimedia/wikipedia"
            except Exception as e: print(f"   (Wikipedia {lang} falló: {type(e).__name__})")
        if got is None and hay_red("https://raw.githubusercontent.com"):
            try:
                mirror = _espejo_nltk(n); got = [d for d in mirror if d["lang"] == lang]; detail[lang] = "espejo:nltk-europarl"
            except Exception as e: print(f"   (espejo {lang} falló: {type(e).__name__})")
        if got is None: got, detail[lang] = [d for d in _sintetico(n) if d["lang"] == lang], "sintetico"
        docs += got
    code = None
    if hf_ok:
        try: code, detail["code"] = _hf_codesearchnet(n), "hf:code_search_net"
        except Exception as e: print(f"   (CodeSearchNet falló: {type(e).__name__})")
    if code is None:
        code = _codigo_stdlib(n); detail["code"] = "espejo:stdlib-python"
        if len(code) < 10: code, detail["code"] = [d for d in _sintetico(n) if d["lang"] == "code"], "sintetico"
    docs += code
    queries, qrels = _mini_golden(docs)
    real = all(not v.startswith("sintetico") for v in detail.values())
    if not real: print("AVISO: parte del corpus es SINTÉTICA (sin acceso a los datasets). Las cifras absolutas cambiarán con datos reales.")
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real" if real else "synthetic", "detail": detail}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs = corpus["docs"]
print("fuente:", corpus["source"], corpus["detail"])
by_lang = {l: [d["text"] for d in docs if d["lang"] == l] for l in ["es", "en", "code"]}
for l, ts in by_lang.items():
    print(f"{l:>4}: {len(ts):4d} docs | {sum(len(t) for t in ts)/1e3:8.1f} k caracteres | {sum(len(t.encode()) for t in ts)/1e3:8.1f} k bytes")
print("\nEjemplo ES:", by_lang["es"][0][:160].replace("\n", " "))
print("Ejemplo código:\n", by_lang["code"][0][:200])

In [ ]:
# Frases paralelas (tipo FLORES): oficial -> espejo -> mini-conjunto escrito a mano (16 frases x 12 idiomas)
_PARALELO_MANO = {
 "en": ["The search engine returns the most relevant documents for each query.", "Please restart the server before installing the new version.",
        "The model was trained on a large collection of texts in several languages.", "Every user must change their password every ninety days.",
        "The function receives a list of numbers and returns their average.", "Yesterday it rained all day, so we stayed at home.",
        "The company opened three new offices in Europe this year.", "Reading the documentation carefully saves a lot of time.",
        "The meeting will begin at nine o'clock in the morning.", "Our system supports more than forty languages and several file formats.",
        "The children are playing in the park near the river.", "To install the library, run the command shown below.",
        "The results show that the new method is faster and more accurate.", "Water boils at one hundred degrees Celsius at sea level.",
        "She asked whether the database could be restored from a backup.", "Errors must be recorded in a log file for later analysis."],
 "es": ["El motor de búsqueda devuelve los documentos más relevantes para cada consulta.", "Reinicie el servidor antes de instalar la nueva versión.",
        "El modelo se entrenó con una gran colección de textos en varios idiomas.", "Cada usuario debe cambiar su contraseña cada noventa días.",
        "La función recibe una lista de números y devuelve su promedio.", "Ayer llovió todo el día, así que nos quedamos en casa.",
        "La empresa abrió tres oficinas nuevas en Europa este año.", "Leer la documentación con atención ahorra mucho tiempo.",
        "La reunión comenzará a las nueve de la mañana.", "Nuestro sistema admite más de cuarenta idiomas y varios formatos de archivo.",
        "Los niños juegan en el parque cerca del río.", "Para instalar la biblioteca, ejecute el comando que se muestra a continuación.",
        "Los resultados muestran que el nuevo método es más rápido y más preciso.", "El agua hierve a cien grados Celsius al nivel del mar.",
        "Ella preguntó si la base de datos se podía restaurar desde una copia de seguridad.", "Los errores deben registrarse en un archivo de registro para su análisis posterior."],
 "fr": ["Le moteur de recherche renvoie les documents les plus pertinents pour chaque requête.", "Veuillez redémarrer le serveur avant d'installer la nouvelle version.",
        "Le modèle a été entraîné sur une grande collection de textes dans plusieurs langues.", "Chaque utilisateur doit changer son mot de passe tous les quatre-vingt-dix jours.",
        "La fonction reçoit une liste de nombres et renvoie leur moyenne.", "Hier, il a plu toute la journée, alors nous sommes restés à la maison.",
        "L'entreprise a ouvert trois nouveaux bureaux en Europe cette année.", "Lire attentivement la documentation fait gagner beaucoup de temps.",
        "La réunion commencera à neuf heures du matin.", "Notre système prend en charge plus de quarante langues et plusieurs formats de fichier.",
        "Les enfants jouent dans le parc près de la rivière.", "Pour installer la bibliothèque, exécutez la commande ci-dessous.",
        "Les résultats montrent que la nouvelle méthode est plus rapide et plus précise.", "L'eau bout à cent degrés Celsius au niveau de la mer.",
        "Elle a demandé si la base de données pouvait être restaurée à partir d'une sauvegarde.", "Les erreurs doivent être enregistrées dans un fichier journal pour une analyse ultérieure."],
 "de": ["Die Suchmaschine liefert für jede Anfrage die relevantesten Dokumente.", "Bitte starten Sie den Server neu, bevor Sie die neue Version installieren.",
        "Das Modell wurde mit einer großen Sammlung von Texten in mehreren Sprachen trainiert.", "Jeder Benutzer muss sein Passwort alle neunzig Tage ändern.",
        "Die Funktion erhält eine Liste von Zahlen und gibt deren Durchschnitt zurück.", "Gestern hat es den ganzen Tag geregnet, deshalb sind wir zu Hause geblieben.",
        "Das Unternehmen hat dieses Jahr drei neue Büros in Europa eröffnet.", "Wer die Dokumentation sorgfältig liest, spart viel Zeit.",
        "Die Besprechung beginnt um neun Uhr morgens.", "Unser System unterstützt mehr als vierzig Sprachen und mehrere Dateiformate.",
        "Die Kinder spielen im Park in der Nähe des Flusses.", "Um die Bibliothek zu installieren, führen Sie den unten gezeigten Befehl aus.",
        "Die Ergebnisse zeigen, dass die neue Methode schneller und genauer ist.", "Wasser kocht auf Meereshöhe bei hundert Grad Celsius.",
        "Sie fragte, ob die Datenbank aus einer Sicherung wiederhergestellt werden könne.", "Fehler müssen für die spätere Analyse in einer Protokolldatei festgehalten werden."],
 "pt": ["O mecanismo de busca devolve os documentos mais relevantes para cada consulta.", "Reinicie o servidor antes de instalar a nova versão.",
        "O modelo foi treinado com uma grande coleção de textos em vários idiomas.", "Cada usuário deve trocar a senha a cada noventa dias.",
        "A função recebe uma lista de números e devolve a média deles.", "Ontem choveu o dia todo, por isso ficamos em casa.",
        "A empresa abriu três novos escritórios na Europa este ano.", "Ler a documentação com atenção poupa muito tempo.",
        "A reunião começará às nove horas da manhã.", "Nosso sistema suporta mais de quarenta idiomas e vários formatos de arquivo.",
        "As crianças estão brincando no parque perto do rio.", "Para instalar a biblioteca, execute o comando mostrado abaixo.",
        "Os resultados mostram que o novo método é mais rápido e mais preciso.", "A água ferve a cem graus Celsius ao nível do mar.",
        "Ela perguntou se o banco de dados poderia ser restaurado a partir de um backup.", "Os erros devem ser registrados em um arquivo de log para análise posterior."],
}

In [ ]:
_PARALELO_MANO.update({
 "ru": ["Поисковая система возвращает наиболее релевантные документы для каждого запроса.", "Перезапустите сервер перед установкой новой версии.",
        "Модель была обучена на большой коллекции текстов на нескольких языках.", "Каждый пользователь должен менять пароль каждые девяносто дней.",
        "Функция принимает список чисел и возвращает их среднее значение.", "Вчера целый день шёл дождь, поэтому мы остались дома.",
        "В этом году компания открыла три новых офиса в Европе.", "Внимательное чтение документации экономит много времени.",
        "Совещание начнётся в девять часов утра.", "Наша система поддерживает более сорока языков и несколько форматов файлов.",
        "Дети играют в парке возле реки.", "Чтобы установить библиотеку, выполните команду, приведённую ниже.",
        "Результаты показывают, что новый метод быстрее и точнее.", "Вода кипит при ста градусах Цельсия на уровне моря.",
        "Она спросила, можно ли восстановить базу данных из резервной копии.", "Ошибки необходимо записывать в файл журнала для последующего анализа."],
 "zh": ["搜索引擎会为每个查询返回最相关的文档。", "请在安装新版本之前重新启动服务器。", "该模型是在多种语言的大量文本上训练的。", "每位用户必须每九十天更换一次密码。",
        "该函数接收一个数字列表并返回它们的平均值。", "昨天下了一整天的雨，所以我们待在家里。", "今年公司在欧洲新开了三个办事处。", "仔细阅读文档可以节省很多时间。",
        "会议将于上午九点开始。", "我们的系统支持四十多种语言和多种文件格式。", "孩子们正在河边的公园里玩耍。", "要安装该库，请运行下面显示的命令。",
        "结果表明，新方法更快、更准确。", "在海平面，水在一百摄氏度时沸腾。", "她问是否可以从备份中恢复数据库。", "错误必须记录在日志文件中，以便日后分析。"],
 "ja": ["検索エンジンは、各クエリに対して最も関連性の高い文書を返します。", "新しいバージョンをインストールする前に、サーバーを再起動してください。", "このモデルは、複数の言語による大量のテキストで訓練されました。",
        "すべてのユーザーは九十日ごとにパスワードを変更しなければなりません。", "この関数は数値のリストを受け取り、その平均を返します。", "昨日は一日中雨が降っていたので、私たちは家にいました。",
        "その会社は今年、ヨーロッパに三つの新しいオフィスを開きました。", "ドキュメントを丁寧に読むと、多くの時間を節約できます。", "会議は午前九時に始まります。",
        "私たちのシステムは、四十以上の言語と複数のファイル形式に対応しています。", "子どもたちは川の近くの公園で遊んでいます。", "ライブラリをインストールするには、以下のコマンドを実行してください。",
        "結果は、新しい方法のほうが速く、より正確であることを示しています。", "水は海面の高さで摂氏百度で沸騰します。", "彼女は、データベースをバックアップから復元できるかどうか尋ねました。",
        "エラーは、後で分析できるようにログファイルに記録する必要があります。"],
 "ko": ["검색 엔진은 각 질의에 대해 가장 관련성이 높은 문서를 반환합니다.", "새 버전을 설치하기 전에 서버를 다시 시작하십시오.", "이 모델은 여러 언어로 된 방대한 텍스트 모음으로 학습되었습니다.",
        "모든 사용자는 구십 일마다 비밀번호를 변경해야 합니다.", "이 함수는 숫자 목록을 받아 그 평균을 반환합니다.", "어제는 하루 종일 비가 와서 우리는 집에 있었습니다.",
        "그 회사는 올해 유럽에 새 사무실 세 곳을 열었습니다.", "문서를 꼼꼼히 읽으면 많은 시간을 절약할 수 있습니다.", "회의는 오전 아홉 시에 시작됩니다.",
        "저희 시스템은 마흔 개가 넘는 언어와 여러 파일 형식을 지원합니다.", "아이들이 강 근처 공원에서 놀고 있습니다.", "라이브러리를 설치하려면 아래에 표시된 명령을 실행하십시오.",
        "결과는 새로운 방법이 더 빠르고 더 정확하다는 것을 보여 줍니다.", "물은 해수면에서 섭씨 백 도에 끓습니다.", "그녀는 데이터베이스를 백업에서 복원할 수 있는지 물었습니다.",
        "오류는 나중에 분석할 수 있도록 로그 파일에 기록해야 합니다."],
 "ar": ["يعيد محرك البحث أكثر المستندات صلة بكل استعلام.", "يرجى إعادة تشغيل الخادم قبل تثبيت الإصدار الجديد.", "تم تدريب النموذج على مجموعة كبيرة من النصوص بعدة لغات.",
        "يجب على كل مستخدم تغيير كلمة المرور كل تسعين يومًا.", "تستقبل الدالة قائمة من الأرقام وتعيد متوسطها.", "أمطرت السماء طوال أمس، لذلك بقينا في المنزل.",
        "افتتحت الشركة ثلاثة مكاتب جديدة في أوروبا هذا العام.", "قراءة الوثائق بعناية توفر الكثير من الوقت.", "سيبدأ الاجتماع في الساعة التاسعة صباحًا.",
        "يدعم نظامنا أكثر من أربعين لغة وعدة تنسيقات للملفات.", "الأطفال يلعبون في الحديقة القريبة من النهر.", "لتثبيت المكتبة، شغّل الأمر الموضح أدناه.",
        "تُظهر النتائج أن الطريقة الجديدة أسرع وأكثر دقة.", "يغلي الماء عند مئة درجة مئوية عند مستوى سطح البحر.", "سألت عما إذا كان بالإمكان استعادة قاعدة البيانات من نسخة احتياطية.",
        "يجب تسجيل الأخطاء في ملف سجل لتحليلها لاحقًا."],
 "hi": ["खोज इंजन हर क्वेरी के लिए सबसे प्रासंगिक दस्तावेज़ लौटाता है।", "नया संस्करण इंस्टॉल करने से पहले कृपया सर्वर को पुनः आरंभ करें।", "इस मॉडल को कई भाषाओं के पाठों के एक बड़े संग्रह पर प्रशिक्षित किया गया था।",
        "हर उपयोगकर्ता को हर नब्बे दिन में अपना पासवर्ड बदलना होगा।", "यह फ़ंक्शन संख्याओं की एक सूची लेता है और उनका औसत लौटाता है।", "कल पूरे दिन बारिश हुई, इसलिए हम घर पर ही रहे।",
        "कंपनी ने इस साल यूरोप में तीन नए कार्यालय खोले।", "दस्तावेज़ीकरण को ध्यान से पढ़ने से बहुत समय बचता है।", "बैठक सुबह नौ बजे शुरू होगी।",
        "हमारा सिस्टम चालीस से अधिक भाषाओं और कई फ़ाइल प्रारूपों का समर्थन करता है।", "बच्चे नदी के पास वाले पार्क में खेल रहे हैं।", "लाइब्रेरी इंस्टॉल करने के लिए नीचे दिखाया गया कमांड चलाएँ।",
        "परिणाम दिखाते हैं कि नई विधि अधिक तेज़ और अधिक सटीक है।", "समुद्र तल पर पानी सौ डिग्री सेल्सियस पर उबलता है।", "उसने पूछा कि क्या डेटाबेस को बैकअप से बहाल किया जा सकता है।",
        "बाद में विश्लेषण के लिए त्रुटियों को लॉग फ़ाइल में दर्ज किया जाना चाहिए।"],
 "tr": ["Arama motoru her sorgu için en alakalı belgeleri döndürür.", "Yeni sürümü yüklemeden önce lütfen sunucuyu yeniden başlatın.", "Model, birçok dilde yazılmış büyük bir metin koleksiyonuyla eğitildi.",
        "Her kullanıcı parolasını doksan günde bir değiştirmelidir.", "İşlev bir sayı listesi alır ve bunların ortalamasını döndürür.", "Dün bütün gün yağmur yağdı, bu yüzden evde kaldık.",
        "Şirket bu yıl Avrupa'da üç yeni ofis açtı.", "Belgeleri dikkatle okumak çok zaman kazandırır.", "Toplantı sabah saat dokuzda başlayacak.",
        "Sistemimiz kırktan fazla dili ve birçok dosya biçimini destekliyor.", "Çocuklar nehrin yakınındaki parkta oynuyor.", "Kitaplığı yüklemek için aşağıda gösterilen komutu çalıştırın.",
        "Sonuçlar, yeni yöntemin daha hızlı ve daha doğru olduğunu gösteriyor.", "Su, deniz seviyesinde yüz santigrat derecede kaynar.", "Veritabanının bir yedekten geri yüklenip yüklenemeyeceğini sordu.",
        "Hatalar, daha sonra analiz edilebilmesi için bir günlük dosyasına kaydedilmelidir."],
})

In [ ]:
FLORES_CODES = {"en": "eng_Latn", "es": "spa_Latn", "fr": "fra_Latn", "de": "deu_Latn", "pt": "por_Latn", "ru": "rus_Cyrl", "zh": "zho_Hans",
                "ja": "jpn_Jpan", "ko": "kor_Hang", "ar": "arb_Arab", "hi": "hin_Deva", "tr": "tur_Latn"}

def carga_paralelo(fast_dev: bool = True, langs=None) -> dict:
    """Frases paralelas. Cadena: FLORES+ (HF, puede pedir aceptar términos y token) -> opus-100 en-es -> mini-conjunto a mano.
    Devuelve {"langs": {lang: [frases alineadas]}, "source": "flores+"|"opus-100"|"mano", "n": int}."""
    langs = langs or list(FLORES_CODES)
    n = 100 if fast_dev else 997
    if hay_red("https://huggingface.co"):
        try:
            from datasets import load_dataset
            out = {l: [ex["text"] for ex in load_dataset("openlanguagedata/flores_plus", FLORES_CODES[l], split="devtest")][:n] for l in langs}
            return {"langs": out, "source": "flores+", "n": len(out["en"])}
        except Exception as e:
            print("   (FLORES+ no disponible:", type(e).__name__, "-> intento opus-100 en-es)")
        try:
            from datasets import load_dataset
            ds = load_dataset("Helsinki-NLP/opus-100", "en-es", split="test")
            tr = [ex["translation"] for ex in ds][:n]
            return {"langs": {"en": [t["en"] for t in tr], "es": [t["es"] for t in tr]}, "source": "opus-100", "n": len(tr)}
        except Exception as e:
            print("   (opus-100 no disponible:", type(e).__name__, ")")
    print("AVISO: usando el mini-conjunto paralelo escrito a mano (16 frases). Para cifras fiables usa FLORES+ con acceso a HF.")
    out = {l: _PARALELO_MANO[l] for l in langs if l in _PARALELO_MANO}
    return {"langs": out, "source": "mano", "n": len(out["en"])}

par = carga_paralelo(fast_dev=FAST_DEV_RUN)
print('paralelo:', par['source'], '| frases:', par['n'], '| idiomas:', list(par['langs']))

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out
WORD_RE = re.compile(r"\w+(?:['’]\w+)?|[^\w\s]", re.UNICODE)
def palabras(text: str): return WORD_RE.findall(text)

def metricas_tokenizacion(encode, textos) -> dict:
    """encode: str -> list[str|int]. Devuelve fertilidad (tokens/palabra), bytes/token y caracteres/token."""
    n_tok = n_pal = n_byte = n_chr = 0
    for t in textos:
        n_tok += len(encode(t)); n_pal += len(palabras(t)); n_byte += len(t.encode("utf-8")); n_chr += len(t)
    return {"fertilidad": n_tok / max(n_pal, 1), "bytes_por_token": n_byte / max(n_tok, 1), "chars_por_token": n_chr / max(n_tok, 1), "tokens": n_tok}

In [ ]:
import tiktoken
TIK = {}
for nombre in ["r50k_base", "cl100k_base", "o200k_base"]:
    try:
        TIK[nombre] = tiktoken.get_encoding(nombre)     # 1ª vez: descarga de openaipublic.blob.core.windows.net
    except Exception as e:
        print(f"(no pude cargar {nombre}: {type(e).__name__}; tras un proxy usa TIKTOKEN_CACHE_DIR). Se omitirá en los análisis.")
print({n: e.n_vocab for n, e in TIK.items()})

In [ ]:
import pandas as pd
from tokenizers import Tokenizer, models, pre_tokenizers, normalizers, trainers, decoders, Regex
from tokenizers.pre_tokenizers import ByteLevel
rng = random.Random(SEED); train_docs, test_docs = [], []
for lang in ["es", "en", "code"]:
    pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool); cut = int(0.8 * len(pool))
    train_docs += pool[:cut]; test_docs += pool[cut:]
test_by_lang = {l: [d["text"] for d in test_docs if d["lang"] == l] for l in ["es", "en", "code"]}

def entrena_bpe_bytes(textos, vocab_size: int) -> Tokenizer:
    """Byte-level BPE (regex GPT-2). Es el 'tokenizer propio' de Políglota (módulos 02-03)."""
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(PAT), behavior="isolated"), ByteLevel(add_prefix_space=False, use_regex=False)])
    t.decoder = decoders.ByteLevel()
    t.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=ByteLevel.alphabet(), special_tokens=["<|endoftext|>"]))
    return t

def corpus_mezcla(prop: dict, total_chars: int, semilla: int = SEED) -> list:
    """Muestrea documentos de train por idioma hasta total_chars*prop[idioma] caracteres."""
    r = random.Random(semilla); out = []
    for lang, p in prop.items():
        pool = [d["text"] for d in train_docs if d["lang"] == lang]; acc = 0
        while acc < p * total_chars:
            t = r.choice(pool); out.append(t); acc += len(t)
    r.shuffle(out); return out

TOTAL = 400_000 if FAST_DEV_RUN else 4_000_000
TRAFICO = {"es": 0.5, "en": 0.3, "code": 0.2}          # supuesto de producto
LAMBDA = 3.0                                           # penalización de vocabulario (tokens/KB por cada 1 k de V)
natural = {"en": 0.80, "es": 0.15, "code": 0.05}
suav = {k: v ** 0.3 / sum(u ** 0.3 for u in natural.values()) for k, v in natural.items()}
MEZCLAS = {"natural": natural, "suavizada": suav, "equilibrada": {"en": 1 / 3, "es": 1 / 3, "code": 1 / 3}, "sesgo_es": {"en": 0.2, "es": 0.6, "code": 0.2}}
VS = [2000, 4000, 8000] if FAST_DEV_RUN else [8000, 16000, 32000]
def intenta(f, *a, **k):
    try: return f(*a, **k)
    except NotImplementedError as e: print("⏳ pendiente:", e); return None
REG = {f"tiktoken {n.replace('_base','')}": (lambda s, e=e: e.encode_ordinary(s)) for n, e in TIK.items()}
REG["BPE propio (natural)"] = (lambda tk: (lambda s: tk.encode(s).ids))(entrena_bpe_bytes(corpus_mezcla(natural, TOTAL), VS[1]))
print(list(REG))

## Paso 1 · Benchmark de paridad

**TODO.** Implementa `benchmark(registro, par)` → `DataFrame` con una fila por (tokenizer, idioma) y columnas `tokens`, `premium` (= tokens del idioma / tokens del inglés) y `bytes_por_token`. Después `tokens_por_kb(registro)` → DataFrame con tokens por KB de texto de test en `es`, `en` y `code` (usa `test_by_lang`).

<details><summary>🪜 Pista 1</summary>

Para cada idioma suma `len(enc(frase))` sobre todas las frases paralelas; la prima divide entre el total del inglés. `bytes_por_token` = bytes UTF-8 totales / tokens totales.
</details>
<details><summary>🪜 Pista 2</summary>

`tot = {l: sum(len(enc(s)) for s in par["langs"][l]) for l in par["langs"]}`; `byt = {l: sum(len(s.encode()) for s in par["langs"][l]) ...}`; en `tokens_por_kb`: `1000 * tokens / bytes` por idioma sobre `test_by_lang[l]`.
</details>

In [ ]:
def benchmark(registro: dict, par: dict) -> pd.DataFrame:
    # TODO: filas {"tokenizer","idioma","tokens","premium","bytes_por_token"}
    raise NotImplementedError("TODO paso 1a")

def tokens_por_kb(registro: dict) -> pd.DataFrame:
    # TODO: índice = tokenizer, columnas = es/en/code (tokens por KB de texto de test)
    raise NotImplementedError("TODO paso 1b")

dp = intenta(benchmark, REG, par); tk_kb = intenta(tokens_por_kb, REG)
if dp is not None: print(dp.pivot_table(index="tokenizer", columns="idioma", values="premium").round(2).to_string())
if tk_kb is not None: print(tk_kb.round(1))

## Paso 2 · Candidatos de vocabulario

**TODO.** `evalua_candidato(tok)` devuelve `{"es": tokens/KB, "en": ..., "code": ...}` sobre el test; `entrena_candidato(prop, V)` entrena un BPE byte-level con el corpus `corpus_mezcla(prop, TOTAL)` y tamaño `V`.

<details><summary>🪜 Pista 1</summary>

Son dos líneas cada una, reutilizando `entrena_bpe_bytes`, `corpus_mezcla` y `test_by_lang`; en tokens/KB usa `1000 * tokens / bytes`.
</details>

In [ ]:
def evalua_candidato(tok: Tokenizer) -> dict:
    # TODO: tokens por KB en es/en/code (test)
    raise NotImplementedError("TODO paso 2a")

def entrena_candidato(prop: dict, V: int) -> Tokenizer:
    # TODO: BPE byte-level con corpus de la mezcla `prop` y tamaño V
    raise NotImplementedError("TODO paso 2b")

## Paso 3 · Selección por coste

**TODO.** `selecciona(mezclas, vs)` entrena todas las combinaciones, calcula para cada una
$J=\sum_\ell w_\ell\cdot\text{tokens/KB}_\ell+\lambda\,V/1000$ (con `TRAFICO` y `LAMBDA`) y la **peor razón** entre idiomas $\max_\ell \text{tpk}_\ell/\min_\ell \text{tpk}_\ell$ (equidad), y devuelve un `DataFrame` con columnas `mezcla, V, tpk_es, tpk_en, tpk_code, J, desbalance`, ordenado por `J`.

<details><summary>🪜 Pista 1</summary>

Doble bucle `for nombre, prop in mezclas.items(): for V in vs:` + `evalua_candidato(entrena_candidato(prop, V))`.
</details>
<details><summary>🪜 Pista 2</summary>

`J = sum(TRAFICO[l] * t[l] for l in t) + LAMBDA * V / 1e3`; `desbalance = max(t.values()) / min(t.values())`.
</details>

In [ ]:
def selecciona(mezclas: dict, vs: list) -> pd.DataFrame:
    # TODO: barrido mezclas x V; columnas mezcla, V, tpk_es, tpk_en, tpk_code, J, desbalance; ordenado por J
    raise NotImplementedError("TODO paso 3")

cand = intenta(selecciona, MEZCLAS, VS)
if cand is not None: print(cand.round(2).to_string(index=False))

## Paso 4 · La receta

**TODO.** Escribe `poliglota_vocab.json` con `{"mezcla": ..., "V": ..., "tokens_por_kb": {...}, "J": ..., "justificacion": "..."}` (justificación ≥ 300 caracteres con **≥ 3 cifras** y **un riesgo**: p. ej. qué idioma pierde con tu elección y qué harías si el tráfico cambia).

In [ ]:
receta = {}      # TODO: dict con la receta (ver enunciado)

## Rúbrica (autoevaluación)

Referencia medida (corpus espejo, `FAST_DEV_RUN=True`): prima del hindi con cl100k ≈ 4.9 (umbral > 3.0); mejor candidato J ≈ 269 frente a ≈ 303 de la mezcla natural con V = 4 000 (−11 %; umbral −8 %); desbalance del mejor ≈ 1.2 (umbral ≤ 1.35).

In [ ]:
def rubrica():
    R = []; ok = lambda nombre, cond, pts: R.append((nombre, bool(cond), pts))
    ok("1a · benchmark devuelve prima 1.0 para el inglés en todos los tokenizers", dp is not None and np.allclose(dp[dp.idioma == "en"]["premium"], 1.0), 2)
    if dp is not None and "tiktoken o200k" in set(dp.tokenizer) and "tiktoken cl100k" in set(dp.tokenizer):
        P = dp.pivot_table(index="tokenizer", columns="idioma", values="premium")
        ok("1a · o200k reduce la prima media (no-inglés) frente a cl100k", P.loc["tiktoken o200k"].mean() < P.loc["tiktoken cl100k"].mean(), 2)
        ok("1a · la prima del hindi con cl100k > UMBRAL_HI (n pequeño: orientativo)", P.loc["tiktoken cl100k"].get("hi", 0) > UMBRALES["hi_cl100k"], 1)
    else:
        ok("1a · tiktoken disponible (o200k y cl100k)", False, 3)
    ok("1b · tokens por KB calculados para es/en/code", tk_kb is not None and {"es", "en", "code"} <= set(tk_kb.columns), 2)
    ok("3 · el barrido devuelve mezclas × V ordenado por J", cand is not None and len(cand) == len(MEZCLAS) * len(VS) and cand["J"].is_monotonic_increasing, 2)
    if cand is not None and len(cand):
        mejor = cand.iloc[0]; base = cand[(cand.mezcla == "natural") & (cand.V == VS[1])]["J"].iloc[0]
        ok(f"3 · el mejor candidato mejora a 'natural'@V={VS[1]} en ≥ {UMBRALES['mejora_J']:.0%} de J", mejor["J"] <= base * (1 - UMBRALES["mejora_J"]), 2)
        ok("3 · el mejor candidato NO es la mezcla 'natural' (con este tráfico 50/30/20)", mejor["mezcla"] != "natural", 1)
        ok(f"3 · desbalance del mejor ≤ {UMBRALES['desbalance']}", mejor["desbalance"] <= UMBRALES["desbalance"], 1)
    else:
        ok("3 · candidatos", False, 4)
    ok("4 · receta con mezcla, V y J", {"mezcla", "V", "J"} <= set(receta), 1)
    j = str(receta.get("justificacion", ""))
    ok("4 · justificación ≥ 300 caracteres con ≥ 3 cifras", len(j) >= 300 and len(re.findall(r"\d+[.,]?\d*", j)) >= 3, 2)
    tot = sum(p for _, _, p in R); got = sum(p for _, c, p in R if c)
    for nombre, c, p in R: print(("✅" if c else "⬜"), f"[{p}]", nombre)
    print(f"\nPuntuación: {got}/{tot}")
UMBRALES = {"hi_cl100k": 3.0, "mejora_J": 0.08, "desbalance": 1.35}
rubrica()

## ⛔ SPOILER — intenta resolverlo primero

Debajo está la solución de referencia completa; al ejecutarla se redefinen las funciones y la rúbrica final puntúa la referencia.

In [ ]:
def benchmark(registro: dict, par: dict) -> pd.DataFrame:
    langs = list(par["langs"]); filas = []
    for nombre, enc in registro.items():
        tot = {l: sum(len(enc(s)) for s in par["langs"][l]) for l in langs}
        byt = {l: sum(len(s.encode("utf-8")) for s in par["langs"][l]) for l in langs}
        for l in langs: filas.append({"tokenizer": nombre, "idioma": l, "tokens": tot[l], "premium": tot[l] / tot["en"], "bytes_por_token": byt[l] / tot[l]})
    return pd.DataFrame(filas)

def tokens_por_kb(registro: dict) -> pd.DataFrame:
    out = {}
    for nombre, enc in registro.items():
        out[nombre] = {l: 1000 * sum(len(enc(t)) for t in test_by_lang[l]) / sum(len(t.encode()) for t in test_by_lang[l]) for l in ["es", "en", "code"]}
    return pd.DataFrame(out).T

dp = benchmark(REG, par); tk_kb = tokens_por_kb(REG)
print(dp.pivot_table(index="tokenizer", columns="idioma", values="premium").round(2).to_string()); print(tk_kb.round(1))

def evalua_candidato(tok: Tokenizer) -> dict:
    return {l: 1000 * sum(len(tok.encode(t).ids) for t in test_by_lang[l]) / sum(len(t.encode()) for t in test_by_lang[l]) for l in ["es", "en", "code"]}

def entrena_candidato(prop: dict, V: int) -> Tokenizer:
    return entrena_bpe_bytes(corpus_mezcla(prop, TOTAL), V)

def selecciona(mezclas: dict, vs: list) -> pd.DataFrame:
    filas = []
    for nombre, prop in mezclas.items():
        for V in vs:
            t = evalua_candidato(entrena_candidato(prop, V))
            J = sum(TRAFICO[l] * t[l] for l in t) + LAMBDA * V / 1e3
            filas.append({"mezcla": nombre, "V": V, "tpk_es": t["es"], "tpk_en": t["en"], "tpk_code": t["code"], "J": J, "desbalance": max(t.values()) / min(t.values())})
    return pd.DataFrame(filas).sort_values("J").reset_index(drop=True)

cand = selecciona(MEZCLAS, VS); print(cand.round(2).to_string(index=False))

In [ ]:
mejor = cand.iloc[0]; nat = cand[(cand.mezcla == "natural") & (cand.V == VS[1])].iloc[0]
receta = {"mezcla": mejor["mezcla"], "V": int(mejor["V"]), "J": float(mejor["J"]),
          "tokens_por_kb": {l: float(mejor[f"tpk_{l}"]) for l in ["es", "en", "code"]},
          "justificacion": (f"Elijo la mezcla '{mejor['mezcla']}' con V={int(mejor['V'])}: J={mejor['J']:.1f} frente a {nat['J']:.1f} de la mezcla natural 80/15/5 con V={VS[1]} "
                            f"({1 - mejor['J'] / nat['J']:.0%} menos). Con el tráfico 50 % ES / 30 % EN / 20 % código el español necesita {mejor['tpk_es']:.0f} tokens/KB (frente a {nat['tpk_es']:.0f} con la natural). "
                            f"Riesgo: el inglés pasa de {nat['tpk_en']:.0f} a {mejor['tpk_en']:.0f} tokens/KB y no se evaluó ningún idioma fuera de ES/EN/código (el BPE propio cuesta casi un token por byte en ruso/hindi/árabe según el benchmark de paridad); "
                            f"si el tráfico cambia o se añaden idiomas hay que reentrenar con λ y pesos nuevos antes de congelar el vocabulario del embedder.")}
import json; json.dump(receta, open("poliglota_vocab.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in receta.items() if k != "justificacion"}, indent=1), "| justificación:", len(receta["justificacion"]), "caracteres")
rubrica()

## Retos extra (nivel experto)

1. **Equidad explícita.** Añade a $J$ un término de equidad (p. ej. $\mu\cdot(\text{peor prima}-1)$) y mira cómo cambia la mezcla ganadora; ¿qué $\mu$ iguala ES y EN?
2. **Más idiomas.** Con `carga_paralelo` (FLORES+ si tienes acceso) añade pt/fr/de a la mezcla y mide la prima de hi/ar/ja con y sin ellos.
3. **SuperBPE para Políglota.** Reutiliza la implementación de la lección y compara bytes/token y el nDCG@10 de BM25 (proyecto 03) con BPE a igual $V$.
4. **Dígitos.** Entrena con dígitos sueltos vs. grupos de 3 (cambia la regex) y mide tokens/KB en un corpus numérico (facturas, logs).
5. **Plantilla de chat de Políglota.** Diseña los tokens especiales (roles, `<|doc|>`, `<|code|>`) y escribe un test que verifique que servir y entrenar producen los mismos ids.

## Reflexión (producción y Políglota)

- ¿Qué métrica de tokenización pondrías en el dashboard de producción (por idioma) y qué umbral dispararía una alerta?
- Si un cliente grande exige soporte de japonés/hindi, ¿cuál es tu plan *sin* reentrenar el embedder? (Pista: módulo 07, tokenizer transfer.)
- ¿Cómo repartirías el presupuesto de parámetros entre vocabulario y capas para el embedder pequeño de Políglota?